# exp045: exp029 R3 + Chunk Filter Ablation (l1 single fold)

**目的**: chunk filter (max_prob > 0.2 で chunk drop) の効果を実測。
paper 256 spec は 0.5 だが、BC2026 pseudo の bimodal 分布 (Phase 0 分析) で 0.5 は class coverage 47% 落ち、0.2 が natural cutoff。
exp029 R3 (val_ns22 fold 0 = 0.9177、LB 0.923 5-fold ensemble) を base に、**唯一の variable** として teacher pseudo に chunk filter を適用。

## 設計

- **base**: exp029 R3 spec を完全複製 (eca_nfnet_l1 + exp017 R2 teacher pseudo + Perch distill + 20 ep + same config)
- **唯一の変更**: `CHUNK_FILTER_MAX_PROB = 0.2` を pseudo CSV 読込直後に適用 (BC2026 bimodal 谷、Phase 0 分析推奨)
- **fold**: FOLDS=[0] single fold
- **比較**: 既存 exp029 R3 fold 0 val_ns22 = **0.9177** vs exp045 (with filter) val_ns22

## 期待結果

| Outcome | 解釈 |
|---|---|
| delta > +0.003 | filter 有効、exp029 R3 5-fold 再学習 (15h) で展開 |
| -0.003 ≤ delta ≤ +0.003 | noise floor、無効と判定 |
| delta < -0.003 | filter は drag、学習データが減って悪化 |

## 出力

- Drive: `output/exp045/fold0/r3/ckpt_best_ns22.pth`
- Kaggle Dataset: `maekeso/birdclef2026-exp045-l1-filtered` (新規)

## 前提

- `pseudo_e17.csv` (exp028 e17 pseudo、exp017 R2 teacher) を Drive `kaggle/birdclef2026/pseudo_e17.csv` に DL 済 or NB 内 selective DL
- Perch cache `kaggle/birdclef2026/perch-cache/{emb.npy, meta.csv}` を Drive に存在


In [1]:
# ============================================================
# Cell 1: Setup
# ============================================================
!pip install -q timm librosa soundfile scipy

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, json, shutil, time, subprocess
from pathlib import Path

DRIVE_INPUT_DIR  = Path("/content/drive/MyDrive/kaggle/birdclef2026")
DRIVE_EXP_DIR    = DRIVE_INPUT_DIR / "output" / "exp045"
assert DRIVE_INPUT_DIR.exists()

KJ_CANDIDATES = [DRIVE_INPUT_DIR / "kaggle.json",
                  Path("/content/drive/MyDrive/kaggle.json")]
KJ = next((p for p in KJ_CANDIDATES if p.exists()), None)
if KJ is not None:
    KAGGLE_CFG = Path.home() / ".kaggle"
    KAGGLE_CFG.mkdir(parents=True, exist_ok=True)
    shutil.copy(str(KJ), str(KAGGLE_CFG / "kaggle.json"))
    os.chmod(str(KAGGLE_CFG / "kaggle.json"), 0o600)
    creds = json.loads(KJ.read_text())
    if creds.get("key", "").startswith("KGAT_"):
        os.environ["KAGGLE_API_TOKEN"] = creds["key"]
    print(f"kaggle.json: {KJ}")

LOCAL_DATA = Path("/content/data")
LOCAL_OUT  = Path("/content/output")
LOCAL_DATA.mkdir(parents=True, exist_ok=True)
LOCAL_OUT.mkdir(parents=True, exist_ok=True)


Mounted at /content/drive
kaggle.json: /content/drive/MyDrive/kaggle/birdclef2026/kaggle.json


In [2]:
# ============================================================
# Cell 2: Data DL — competition + Perch cache
# ============================================================
import time, zipfile, subprocess
from kaggle.api.kaggle_api_extended import KaggleApi
from tqdm.auto import tqdm

api = KaggleApi(); api.authenticate()
print("kaggle authenticated")

T0_total = time.time()
TA_DIR = LOCAL_DATA / "train_audio"
TS_DIR = LOCAL_DATA / "train_soundscapes"

need_dl = (
    not TA_DIR.exists() or sum(1 for _ in TA_DIR.rglob("*.ogg")) < 40000 or
    not TS_DIR.exists() or sum(1 for _ in TS_DIR.glob("*.ogg")) < 10000
)
if need_dl:
    print(f"\nDownloading birdclef-2026 (~25GB)...")
    t0 = time.time()
    api.competition_download_files("birdclef-2026", path=str(LOCAL_DATA),
                                    force=False, quiet=False)
    print(f"  DL done in {(time.time()-t0)/60:.1f} min")
    zips = list(LOCAL_DATA.glob("birdclef-2026*.zip"))
    zip_path = zips[0]
    t_extract = time.time()
    with zipfile.ZipFile(zip_path) as zf:
        infos = zf.infolist()
        total_bytes = sum(i.file_size for i in infos)
        pbar = tqdm(total=total_bytes, unit="B", unit_scale=True, unit_divisor=1024,
                    desc="extract", mininterval=1.0)
        for info in infos:
            zf.extract(info, LOCAL_DATA)
            pbar.update(info.file_size)
        pbar.close()
    print(f"  extracted in {(time.time()-t_extract)/60:.1f} min")
    zip_path.unlink()

n_ta = sum(1 for _ in TA_DIR.rglob("*.ogg")) if TA_DIR.exists() else 0
n_ts = sum(1 for _ in TS_DIR.glob("*.ogg")) if TS_DIR.exists() else 0
print(f"  train_audio: {n_ta}, train_soundscapes: {n_ts}")

comp = LOCAL_DATA / "competition"
comp.mkdir(parents=True, exist_ok=True)
for fn in ["train.csv", "taxonomy.csv", "sample_submission.csv", "train_soundscapes_labels.csv"]:
    src_in_root = LOCAL_DATA / fn
    dst = comp / fn
    if src_in_root.exists() and not dst.exists():
        shutil.copy2(str(src_in_root), str(dst))

PERCH_CACHE_DIR = LOCAL_DATA / "perch-cache"
EMB_PATH = PERCH_CACHE_DIR / "emb.npy"
META_PATH = PERCH_CACHE_DIR / "meta.csv"
if not EMB_PATH.exists() or not META_PATH.exists():
    print(f"\nDownloading Perch cache (~1.1GB)...")
    PERCH_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    api.dataset_download_files("maekeso/birdclef2026-perch-emb-cache",
                                path=str(PERCH_CACHE_DIR), unzip=True, quiet=False)
    print(f"  DL done in {(time.time()-t0)/60:.1f} min")
assert EMB_PATH.exists() and META_PATH.exists()
print(f"  emb.npy: {EMB_PATH.stat().st_size/1e9:.2f}GB")

# ★ exp029: teacher pseudo (pseudo_e17.csv) を Kaggle NB output から DL → Drive へ
TEACHER_PSEUDO_DRIVE = DRIVE_INPUT_DIR / "pseudo_e17.csv"
if not TEACHER_PSEUDO_DRIVE.exists():
    print(f"\nDownloading teacher pseudo (pseudo_e17.csv) from Kaggle NB output...")
    t0 = time.time()
    TEACHER_PSEUDO_TMP = LOCAL_DATA / "pseudo_e17_kaggle"
    TEACHER_PSEUDO_TMP.mkdir(parents=True, exist_ok=True)
    try:
        # Selective DL of pseudo_e17.csv only (avoid huge model output)
        api.kernels_output_download_file("maekeso/birdclef2026-exp028-pseudo-e17",
                                         file_name="pseudo_e17.csv",
                                         path=str(TEACHER_PSEUDO_TMP))
        src = TEACHER_PSEUDO_TMP / "pseudo_e17.csv"
        assert src.exists(), f"DL failed: {src}"
        shutil.copy2(str(src), str(TEACHER_PSEUDO_DRIVE))
        print(f"  DL done in {(time.time()-t0)/60:.1f} min, saved to {TEACHER_PSEUDO_DRIVE}")
    except Exception as e:
        # Fallback: download all output (heavier)
        print(f"  selective DL failed ({e}), trying full output download...")
        api.kernels_output("maekeso/birdclef2026-exp028-pseudo-e17",
                           path=str(TEACHER_PSEUDO_TMP), force=True)
        src = next(TEACHER_PSEUDO_TMP.rglob("pseudo_e17.csv"), None)
        assert src is not None, "pseudo_e17.csv not found in NB output"
        shutil.copy2(str(src), str(TEACHER_PSEUDO_DRIVE))
        print(f"  fallback DL done in {(time.time()-t0)/60:.1f} min")
print(f"  pseudo_e17.csv: {TEACHER_PSEUDO_DRIVE.stat().st_size/1e6:.1f}MB")

print(f"\n=== Total prep time: {(time.time()-T0_total)/60:.1f} min ===")


kaggle authenticated



100%|██████████| 15.0G/15.0G [06:15<00:00, 42.8MB/s]



  DL done in 6.3 min


extract:   0%|          | 0.00/15.0G [00:00<?, ?B/s]

  extracted in 0.9 min
  train_audio: 35549, train_soundscapes: 10658

Dataset URL: https://www.kaggle.com/datasets/maekeso/birdclef2026-perch-emb-cache


100%|██████████| 971M/971M [00:25<00:00, 40.7MB/s]



  DL done in 0.5 min
  emb.npy: 1.11GB
  pseudo_e17.csv: 409.8MB

=== Total prep time: 7.7 min ===


In [3]:
# ============================================================
# Cell 3: Imports + Config
# ============================================================
import os, time, json, gc, random, math
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.cuda.amp import GradScaler, autocast
import torchaudio
import timm
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, GroupKFold
import warnings
warnings.filterwarnings("ignore")

BASE = LOCAL_DATA / "competition"
TA_DIR = LOCAL_DATA / "train_audio"
TS_DIR = LOCAL_DATA / "train_soundscapes"
TAXO_PATH = BASE / "taxonomy.csv"
TRAIN_CSV = BASE / "train.csv"
SAMPLE_SUB_PATH = BASE / "sample_submission.csv"
LABELS_PATH = BASE / "train_soundscapes_labels.csv"
PERCH_CACHE_DIR = LOCAL_DATA / "perch-cache"
EMB_PATH = PERCH_CACHE_DIR / "emb.npy"
META_PATH = PERCH_CACHE_DIR / "meta.csv"

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

NUM_CLASSES = 234
SR = 32000
TRAIN_DURATION = 5
TRAIN_SAMPLES = SR * TRAIN_DURATION
VAL_SAMPLES = TRAIN_SAMPLES
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 256
FMIN = 20
FMAX = 16000

BACKBONE = "eca_nfnet_l1"   # ★ exp029: l0 → l1 (capacity 増、Option B 検証)
USE_PERCH_DISTILL = True
PERCH_EMBED_DIM = 1536
ALPHA_DISTILL = 1.0

N_FOLDS = 5               # ★ exp029: 5-fold split で分割するが、train するのは fold 0 のみ
FOLDS = [0]               # single fold 学習 (homogenize 回避)、ただし split は 5-fold で行う

# ★ Plan I: 15 ep + NUM_WORKERS=16 で時短重視
# pseudo distill peak ep 4-8、15 ep で buffer +7-11 ep (十分)
# best_ckpt 自動保存で peak 確実捕捉、val 同等期待
# 5-fold 5.5-6h、CU 余裕で Plan B (multi-teacher) も実行可
N_TOTAL_EPOCHS = 20         # ★ exp029 Option C: 15 → 20、warmup 4/20 = 20% で適正
BATCH = 192
LR = 3e-4
MIN_LR = 1e-6
WD = 1e-4
WARMUP_EPOCHS = 4         # ★ exp029: l1 で +1 ep (exp022 同設定)

AUG_PROB = 0.5
AUG_GAIN_DB_RANGE = (-6.0, 6.0)
AUG_NOISE_SNR_DB_RANGE = (10.0, 30.0)
USE_MIXUP = True
MIXUP_PROB = 0.5
MIXUP_ALPHA = 0.4
MIXUP_HARD = False
FREQ_MASK_PARAM = 25
TIME_MASK_PARAM = 30
NUM_FREQ_MASKS = 2
NUM_TIME_MASKS = 2
MIN_SAMPLE = 20

SHARES_R2 = {"focal": 0.65, "labeled_sc": 0.10, "pseudo_sc": 0.25}   # ★ exp029 Option C: pseudo 0.20 → 0.25、distill 強化

# ★ exp045 Ablation: paper 256 流 chunk filter (max_prob > 0.5 で teacher pseudo の chunk drop)
# CHUNK_FILTER_MAX_PROB = 0.0 で disable (exp029 R3 baseline と同等)
# CHUNK_FILTER_MAX_PROB = 0.2 で paper 256 spec (Sydorskyi 2nd place の primary_label_min_prob)
CHUNK_FILTER_MAX_PROB = 0.2

SOURCE_WEIGHTS = {"focal": 1.0, "focal_missing": 0.0, "labeled_sc": 1.0, "pseudo_sc": 0.5}

NUM_WORKERS = 16        # Plan I: 12 → 16、data loading 並列度 +33%
PERSISTENT_WORKERS = True

SESSION_START = time.time()
MAX_RUNTIME_SEC = 22.0 * 3600

print(f"Backbone: {BACKBONE} | Folds: {FOLDS}")
print(f"R2 epochs: {N_TOTAL_EPOCHS} | warmup: {WARMUP_EPOCHS} | batch: {BATCH} | LR: {LR}")
print(f"Sources: {SHARES_R2}")


Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 102.0 GB
Backbone: eca_nfnet_l1 | Folds: [0]
R2 epochs: 20 | warmup: 4 | batch: 192 | LR: 0.0003
Sources: {'focal': 0.65, 'labeled_sc': 0.1, 'pseudo_sc': 0.25}


In [4]:
# ============================================================
# Cell 4: Load CSVs + Perch cache lookup tables
# ============================================================
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
PRIMARY_LABELS = sample_sub.columns[1:].tolist()
LABEL2IDX = {label: idx for idx, label in enumerate(PRIMARY_LABELS)}
assert len(PRIMARY_LABELS) == NUM_CLASSES

taxonomy = pd.read_csv(TAXO_PATH)
label_to_taxon = dict(zip(taxonomy["primary_label"].astype(str),
                          taxonomy["class_name"].astype(str)))
TAXON_MASKS = {t: np.array([i for i, l in enumerate(PRIMARY_LABELS)
                            if label_to_taxon.get(l, "") == t])
               for t in ["Aves", "Amphibia", "Insecta", "Mammalia", "Reptilia"]}

train_df = pd.read_csv(TRAIN_CSV)
train_df = train_df[train_df["primary_label"].astype(str).isin(LABEL2IDX)].reset_index(drop=True)
train_df["filename"] = train_df["filename"].astype(str)
train_df["exists"] = train_df["filename"].map(lambda fn: (TA_DIR / fn).exists())
train_df = train_df[train_df["exists"]].drop(columns=["exists"]).reset_index(drop=True)
train_df["original_idx"] = np.arange(len(train_df))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
train_df["fold"] = -1
for fold, (_, val_idx) in enumerate(skf.split(train_df, train_df["primary_label"])):
    train_df.loc[val_idx, "fold"] = fold

focal_secondary_labels = {}
for idx, row in train_df.iterrows():
    sec = row.get("secondary_labels", "")
    if pd.isna(sec) or sec in ("", "[]"): continue
    try:
        sec_list = eval(sec) if isinstance(sec, str) else []
    except Exception: continue
    valid = [s for s in sec_list if s in LABEL2IDX]
    if valid:
        focal_secondary_labels[int(row["original_idx"])] = valid

counts = train_df["primary_label"].value_counts()
rare_species = counts[counts < MIN_SAMPLE].index.tolist()
extra_rows = []
for sp in rare_species:
    sp_rows = train_df[train_df["primary_label"] == sp]
    n_copies = int(np.ceil(MIN_SAMPLE / len(sp_rows))) - 1
    for _ in range(n_copies):
        extra_rows.append(sp_rows)
if extra_rows:
    train_df = pd.concat([train_df] + extra_rows, ignore_index=True)

if LABELS_PATH.exists():
    sc_labels_raw = pd.read_csv(LABELS_PATH).drop_duplicates()
    if sc_labels_raw["start"].dtype == object:
        sc_labels_raw["start_sec"] = pd.to_timedelta(sc_labels_raw["start"]).dt.total_seconds().astype(int)
    else:
        sc_labels_raw["start_sec"] = sc_labels_raw["start"].astype(int)
    sc_meta = sc_labels_raw[["filename", "start_sec"]].drop_duplicates().reset_index(drop=True)
    if "site" in sc_labels_raw.columns:
        site_map = sc_labels_raw.groupby("filename")["site"].first().to_dict()
        sc_meta["site"] = sc_meta["filename"].map(site_map).fillna("UNK")
    else:
        sc_meta["site"] = "UNK"
    Y_SC = np.zeros((len(sc_meta), NUM_CLASSES), dtype=np.float32)
    for i, row in sc_meta.iterrows():
        matches = sc_labels_raw[(sc_labels_raw["filename"] == row["filename"]) &
                                 (sc_labels_raw["start_sec"] == row["start_sec"])]
        for _, m in matches.iterrows():
            for lbl in str(m["primary_label"]).split(";"):
                lbl = lbl.strip()
                if lbl in LABEL2IDX:
                    Y_SC[i, LABEL2IDX[lbl]] = 1.0
    sc_files = sc_meta[["filename", "site"]].drop_duplicates().reset_index(drop=True)
    gkf = GroupKFold(n_splits=N_FOLDS)
    sc_files["fold"] = -1
    for fold, (_, val_idx) in enumerate(gkf.split(sc_files, groups=sc_files["filename"])):
        sc_files.loc[sc_files.index[val_idx], "fold"] = fold
    file_to_fold = dict(zip(sc_files["filename"], sc_files["fold"]))
    sc_meta["fold"] = sc_meta["filename"].map(file_to_fold).fillna(-1).astype(int)
    non_s22_mask_sc = (sc_meta["site"].values != "S22")
else:
    sc_meta = pd.DataFrame(columns=["filename", "start_sec", "site", "fold"])
    Y_SC = np.zeros((0, NUM_CLASSES), dtype=np.float32)
    non_s22_mask_sc = np.zeros(0, dtype=bool)

# Perch cache lookup
print(f"\nLoading Perch cache meta.csv...")
meta_df = pd.read_csv(META_PATH)
focal_meta = meta_df[meta_df["source"] == "focal"].reset_index(drop=True)
focal_chunk_lookup = {}
for fn, sub in focal_meta.groupby("filename"):
    focal_chunk_lookup[fn] = sub[["chunk_idx", "row_idx"]].values.astype(np.int32)

ss_meta = meta_df[meta_df["source"] == "ss"].reset_index(drop=True)
ss_lookup = {}
for fn, sub in ss_meta.groupby("filename"):
    base = fn.rsplit(".ogg", 1)[0] if fn.endswith(".ogg") else fn
    for ci, ri in sub[["chunk_idx", "row_idx"]].values:
        ss_lookup[(base, int(ci))] = int(ri)
        ss_lookup[(fn, int(ci))]   = int(ri)
print(f"  focal: {len(focal_chunk_lookup)} files, ss: {len(ss_lookup)} entries")

EMB_MMAP = np.load(str(EMB_PATH), mmap_mode="r")
print(f"  emb shape: {EMB_MMAP.shape}")

print("OK CSVs + Perch cache loaded")



Loading Perch cache meta.csv...
  focal: 35549 files, ss: 255792 entries
  emb shape: (360997, 1536)
OK CSVs + Perch cache loaded


In [5]:
# ============================================================
# Cell 5: Model
# ============================================================
class MelSpecTransform(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel_spec = torchaudio.transforms.MelSpectrogram(
            sample_rate=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
            n_mels=N_MELS, f_min=FMIN, f_max=FMAX, power=2.0,
        )
        self.db_transform = torchaudio.transforms.AmplitudeToDB(top_db=80)
    def forward(self, waveform):
        return self.db_transform(self.mel_spec(waveform))

class SpecAugment(nn.Module):
    def __init__(self):
        super().__init__()
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=FREQ_MASK_PARAM)
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=TIME_MASK_PARAM)
    def forward(self, mel):
        for _ in range(NUM_FREQ_MASKS): mel = self.freq_mask(mel)
        for _ in range(NUM_TIME_MASKS): mel = self.time_mask(mel)
        return mel

class DistillHead(nn.Module):
    def __init__(self, backbone_dim, embed_dim=1536):
        super().__init__()
        self.proj = nn.Linear(backbone_dim, embed_dim)
    def forward(self, feature_map):
        return self.proj(feature_map.mean(dim=[2, 3]))

class GeMFreqPool(nn.Module):
    def __init__(self, p_init=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(float(p_init)))
        self.eps = eps
    def forward(self, x):
        p = self.p.clamp(min=1.0)
        x = x.clamp(min=self.eps).pow(p)
        x = x.mean(dim=2)
        return x.pow(1.0 / p)

class BirdSEDModel(nn.Module):
    def __init__(self, backbone_name=BACKBONE, num_classes=NUM_CLASSES,
                 drop_path_rate=0.1, hidden_dim=512):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name, pretrained=True, in_chans=1,
            num_classes=0, global_pool="", drop_path_rate=drop_path_rate,
        )
        with torch.no_grad():
            n_tf = TRAIN_SAMPLES // HOP_LENGTH + 1
            dummy = torch.randn(1, 1, N_MELS, n_tf)
            feat = self.backbone(dummy)
            self.backbone_dim = feat.shape[1]
        self.gem_freq = GeMFreqPool(p_init=3.0)
        self.dense = nn.Sequential(
            nn.Dropout(0.25),
            nn.Linear(self.backbone_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
        )
        self.att = nn.Conv1d(hidden_dim, num_classes, kernel_size=1, bias=True)
        self.cla = nn.Conv1d(hidden_dim, num_classes, kernel_size=1, bias=True)
        nn.init.xavier_uniform_(self.att.weight)
        nn.init.xavier_uniform_(self.cla.weight)
        self.att.bias.data.fill_(0.)
        self.cla.bias.data.fill_(0.)
        if USE_PERCH_DISTILL:
            self.distill_head = DistillHead(self.backbone_dim, PERCH_EMBED_DIM)

    def forward(self, x, return_framewise=False, return_distill=False):
        h = self.backbone(x)
        distill_emb = None
        if return_distill and hasattr(self, "distill_head"):
            distill_emb = self.distill_head(h)
        h_cls = h.detach() if USE_PERCH_DISTILL else h
        h_cls = self.gem_freq(h_cls)
        h_cls = h_cls.permute(0, 2, 1)
        h_cls = self.dense(h_cls)
        h_cls = h_cls.permute(0, 2, 1)
        norm_att = torch.softmax(torch.tanh(self.att(h_cls)), dim=-1)
        framewise_logits = self.cla(h_cls)
        clip_logits = torch.sum(norm_att * framewise_logits, dim=2)
        fw = framewise_logits.permute(0, 2, 1) if return_framewise else None
        if return_framewise and return_distill:
            return clip_logits, fw, distill_emb
        elif return_framewise:
            return clip_logits, fw
        elif return_distill:
            return clip_logits, distill_emb
        return clip_logits

def make_model():
    m = BirdSEDModel().to(device)
    m = m.to(memory_format=torch.channels_last)
    return m

print("OK model defs")


OK model defs


In [6]:
# ============================================================
# Cell 6: Datasets — cache lookup
# ============================================================
import soundfile as sf
import librosa
from functools import lru_cache

@lru_cache(maxsize=512)
def _load_full_audio_cached(path_str):
    try:
        wav, sr = sf.read(path_str, dtype="float32", always_2d=False)
        if wav.ndim > 1:
            wav = wav.mean(axis=1)
        if sr != SR:
            wav = librosa.resample(wav, orig_sr=sr, target_sr=SR)
        return wav.astype(np.float32)
    except Exception:
        return None

def _load_ogg_chunk(path, chunk_idx, n_samples_target=TRAIN_SAMPLES):
    wav = _load_full_audio_cached(str(path))
    if wav is None:
        return None
    start = chunk_idx * n_samples_target
    end = start + n_samples_target
    if end <= len(wav):
        return wav[start:end].copy()
    out = np.zeros(n_samples_target, dtype=np.float32)
    avail = wav[start:start + n_samples_target] if start < len(wav) else np.array([])
    out[:len(avail)] = avail
    return out

def apply_aug(w):
    if np.random.random() < AUG_PROB:
        w = w * (10 ** (np.random.uniform(*AUG_GAIN_DB_RANGE) / 20))
    if np.random.random() < AUG_PROB:
        sp = (w ** 2).mean()
        if sp > 1e-10:
            w = w + np.random.randn(*w.shape).astype(w.dtype) * np.sqrt(
                sp / (10 ** (np.random.uniform(*AUG_NOISE_SNR_DB_RANGE) / 10)))
    return w


class FocalDS(Dataset):
    def __init__(self, df, l2i, secondary_lookup=None, aug=False):
        self.df = df.reset_index(drop=True)
        self.l2i = l2i
        self.aug = aug
        self.secondary_lookup = secondary_lookup
        self.filenames = self.df["filename"].values
        self.primary = self.df["primary_label"].astype(str).values
        self.original_idx = self.df["original_idx"].values if "original_idx" in self.df.columns else None

    def __len__(self): return len(self.df)

    def _load_chunk_random(self, i):
        fn = self.filenames[i]
        path = TA_DIR / fn
        chunks_info = focal_chunk_lookup.get(fn)
        if chunks_info is None or len(chunks_info) == 0:
            chunk = _load_ogg_chunk(path, 0)
            row_idx = -1
        else:
            ci_idx = np.random.randint(len(chunks_info)) if self.aug else 0
            chunk_idx, row_idx = chunks_info[ci_idx]
            chunk = _load_ogg_chunk(path, int(chunk_idx))
        if chunk is None:
            return None, None, -1
        lb = np.zeros(NUM_CLASSES, dtype=np.float32)
        if self.primary[i] in self.l2i:
            lb[self.l2i[self.primary[i]]] = 1.0
        if self.secondary_lookup is not None and self.original_idx is not None:
            for s in self.secondary_lookup.get(int(self.original_idx[i]), []):
                if s in self.l2i: lb[self.l2i[s]] = 1.0
        return chunk, lb, int(row_idx)

    def __getitem__(self, i):
        ch1, lb1, ri1 = self._load_chunk_random(i)
        if ch1 is None:
            return (torch.zeros(1, TRAIN_SAMPLES), torch.zeros(NUM_CLASSES),
                    torch.zeros(PERCH_EMBED_DIM), torch.ones(NUM_CLASSES),
                    torch.ones(NUM_CLASSES), "focal_missing")
        if USE_MIXUP and self.aug and np.random.random() < MIXUP_PROB:
            for _ in range(3):
                j = np.random.randint(len(self.df))
                ch2, lb2, ri2 = self._load_chunk_random(j)
                if ch2 is not None: break
            if ch2 is not None:
                lam = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)
                ch_mix = (lam * ch1 + (1 - lam) * ch2).astype(np.float32)
                if self.aug: ch_mix = apply_aug(ch_mix)
                lb = np.maximum(lb1, lb2) if MIXUP_HARD else (lam * lb1 + (1 - lam) * lb2)
                emb1 = EMB_MMAP[ri1].astype(np.float32) if ri1 >= 0 else np.zeros(PERCH_EMBED_DIM, dtype=np.float32)
                emb2 = EMB_MMAP[ri2].astype(np.float32) if ri2 >= 0 else np.zeros(PERCH_EMBED_DIM, dtype=np.float32)
                emb_mix = (lam * emb1 + (1 - lam) * emb2).astype(np.float32)
                return (torch.from_numpy(ch_mix).unsqueeze(0),
                        torch.from_numpy(lb.astype(np.float32)),
                        torch.from_numpy(emb_mix),
                        torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "focal")
        if self.aug: ch1 = apply_aug(ch1)
        emb = EMB_MMAP[ri1].astype(np.float32) if ri1 >= 0 else np.zeros(PERCH_EMBED_DIM, dtype=np.float32)
        return (torch.from_numpy(ch1.astype(np.float32)).unsqueeze(0),
                torch.from_numpy(lb1),
                torch.from_numpy(emb),
                torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "focal")


class LabeledSCDS(Dataset):
    def __init__(self, Y, sc_df, aug=False):
        self.Y = Y
        self.df = sc_df.reset_index(drop=True)
        self.aug = aug
        self.filenames = self.df["filename"].astype(str).values
        self.start_secs = self.df["start_sec"].astype(int).values

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        fn = self.filenames[i]
        fn_with_ext = fn if fn.endswith(".ogg") else fn + ".ogg"
        path = TS_DIR / fn_with_ext
        start_sec = int(self.start_secs[i])
        chunk_idx = start_sec // 5
        wav = _load_ogg_chunk(path, chunk_idx)
        if wav is None:
            wav = np.zeros(TRAIN_SAMPLES, dtype=np.float32)
        if self.aug:
            wav = apply_aug(wav)
        row_idx = ss_lookup.get((fn, chunk_idx), ss_lookup.get((fn_with_ext, chunk_idx), -1))
        emb = EMB_MMAP[row_idx].astype(np.float32) if row_idx >= 0 else np.zeros(PERCH_EMBED_DIM, dtype=np.float32)
        return (torch.from_numpy(wav.astype(np.float32)).unsqueeze(0),
                torch.from_numpy(self.Y[i].astype(np.float32)),
                torch.from_numpy(emb),
                torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "labeled_sc")


class PseudoScDS(Dataset):
    def __init__(self, meta_df, Y_soft, audio_dir, aug=False):
        self.meta = meta_df.reset_index(drop=True)
        self.Y = Y_soft.astype(np.float32)
        self.audio_dir = Path(audio_dir)
        self.aug = aug
        self.filenames = self.meta["filename"].values
        self.start_secs = self.meta["start_sec"].values

    def __len__(self): return len(self.meta)

    def __getitem__(self, i):
        fn = str(self.filenames[i])
        fn_with_ext = fn if fn.endswith(".ogg") else fn + ".ogg"
        path = self.audio_dir / fn_with_ext
        start_sec = float(self.start_secs[i])
        chunk_idx = int(start_sec // 5)
        wav = _load_ogg_chunk(path, chunk_idx)
        if wav is None:
            wav = np.zeros(TRAIN_SAMPLES, dtype=np.float32)
        if self.aug:
            wav = apply_aug(wav)
        row_idx = ss_lookup.get((fn, chunk_idx), ss_lookup.get((fn_with_ext, chunk_idx), -1))
        emb = EMB_MMAP[row_idx].astype(np.float32) if row_idx >= 0 else np.zeros(PERCH_EMBED_DIM, dtype=np.float32)
        return (torch.from_numpy(wav.astype(np.float32)).unsqueeze(0),
                torch.from_numpy(self.Y[i]),
                torch.from_numpy(emb),
                torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "pseudo_sc")


class MixSamp(torch.utils.data.Sampler):
    def __init__(self, sizes, names, shares, bs, nst, seed=0):
        self.sizes, self.names, self.bs, self.nst = sizes, names, bs, nst
        self.rng = np.random.default_rng(seed)
        per_src = [max(1, int(round(bs * shares.get(n, 0.0)))) for n in names]
        total = sum(per_src)
        if total != bs:
            per_src[int(np.argmax(per_src))] += (bs - total)
        self.per_src = per_src
        self.offsets = [0]
        for s in sizes[:-1]:
            self.offsets.append(self.offsets[-1] + s)
    def __len__(self): return self.nst
    def __iter__(self):
        for _ in range(self.nst):
            batch = []
            for off, size, n in zip(self.offsets, self.sizes, self.per_src):
                if n <= 0 or size <= 0: continue
                idxs = self.rng.integers(0, size, size=n)
                batch.extend([off + int(i) for i in idxs])
            self.rng.shuffle(batch)
            yield batch

def collate_m(batch):
    return (torch.stack([b[0] for b in batch]),
            torch.stack([b[1] for b in batch]),
            torch.stack([b[2] for b in batch]),
            torch.stack([b[3] for b in batch]),
            torch.stack([b[4] for b in batch]),
            [b[5] for b in batch])

def mk_sw(sr):
    return torch.tensor([SOURCE_WEIGHTS.get(s, 0.0) for s in sr], dtype=torch.float32)

print("OK datasets")


OK datasets


In [7]:
# ============================================================
# Cell 7: Eval helpers
# ============================================================
def compute_macro_auc(y_true, y_pred, mask=None, class_mask=None):
    if mask is not None:
        y_true, y_pred = y_true[mask], y_pred[mask]
    if class_mask is not None:
        y_true, y_pred = y_true[:, class_mask], y_pred[:, class_mask]
    aucs = []
    for c in range(y_true.shape[1]):
        col = y_true[:, c]
        if col.sum() == 0 or col.sum() == len(col): continue
        try:
            aucs.append(roc_auc_score(col, y_pred[:, c]))
        except ValueError: continue
    return (np.mean(aucs) if aucs else float("nan")), len(aucs)


def full_eval(y_true, y_pred, ns22_mask, taxon_masks):
    r = {}
    a, n = compute_macro_auc(y_true, y_pred)
    r["macro_auc_all"] = round(float(a) if not np.isnan(a) else 0.0, 4)
    a, n = compute_macro_auc(y_true, y_pred, mask=ns22_mask)
    r["non_s22_macro"] = round(float(a) if not np.isnan(a) else 0.0, 4)
    per_taxon = {}
    for t, cm in taxon_masks.items():
        a, n = compute_macro_auc(y_true, y_pred, mask=ns22_mask, class_mask=cm)
        per_taxon[t] = round(float(a) if not np.isnan(a) else 0.0, 4)
    r["per_taxon"] = per_taxon
    return r


def _load_val_waveforms(val_sc_df):
    wavs = []
    for _, row in val_sc_df.iterrows():
        fn = str(row["filename"])
        fn_with_ext = fn if fn.endswith(".ogg") else fn + ".ogg"
        start_sec = int(row["start_sec"])
        chunk_idx = start_sec // 5
        wav = _load_ogg_chunk(TS_DIR / fn_with_ext, chunk_idx)
        if wav is None or len(wav) == 0:
            wav = np.zeros(VAL_SAMPLES, dtype=np.float32)
        if len(wav) < VAL_SAMPLES:
            wav = np.pad(wav, (0, VAL_SAMPLES - len(wav)))
        else:
            wav = wav[:VAL_SAMPLES]
        wavs.append(torch.from_numpy(wav.astype(np.float32)).unsqueeze(0))
    return wavs


def _predict_from_waveforms(model, mel_transform, wav_list, batch_size=64):
    model.eval()
    preds_blend = []
    with torch.no_grad():
        for s in range(0, len(wav_list), batch_size):
            batch = torch.stack(wav_list[s:s+batch_size]).to(device)
            mel = mel_transform(batch)
            B = mel.size(0)
            for i in range(B):
                mel[i] = (mel[i] - mel[i].mean()) / (mel[i].std() + 1e-6)
            mel = mel.to(memory_format=torch.channels_last)
            with autocast():
                clip_logits, framewise = model(mel, return_framewise=True)
                frame_max = framewise.max(dim=1).values
                p_clip = torch.sigmoid(clip_logits).float().cpu().numpy()
                p_fmax = torch.sigmoid(frame_max).float().cpu().numpy()
                p_blend = 0.5 * p_clip + 0.5 * p_fmax
            preds_blend.append(p_blend)
    return np.concatenate(preds_blend)

def per_class_distribution(y_true, y_pred, mask=None, class_mask=None):
    """Return per-class AUC array and distribution stats."""
    if mask is not None:
        y_true, y_pred = y_true[mask], y_pred[mask]
    if class_mask is not None:
        y_true, y_pred = y_true[:, class_mask], y_pred[:, class_mask]
    aucs = []
    for c in range(y_true.shape[1]):
        col = y_true[:, c]
        if col.sum() == 0 or col.sum() == len(col):
            aucs.append(np.nan); continue
        try:
            aucs.append(roc_auc_score(col, y_pred[:, c]))
        except ValueError:
            aucs.append(np.nan)
    aucs = np.array(aucs)
    valid = aucs[~np.isnan(aucs)]
    if len(valid) == 0:
        return {"n_valid": 0}
    return {
        "n_valid": len(valid),
        "median": float(np.median(valid)),
        "p25":    float(np.percentile(valid, 25)),
        "p75":    float(np.percentile(valid, 75)),
        "min":    float(valid.min()),
        "max":    float(valid.max()),
        "n_above_0.5": int((valid >= 0.5).sum()),
        "n_above_0.7": int((valid >= 0.7).sum()),
        "n_above_0.9": int((valid >= 0.9).sum()),
        "n_perfect_1.0": int((valid >= 0.999).sum()),
        "auc_array": aucs.tolist(),
    }


def rich_eval(y_true, y_pred, ns22_mask, taxon_masks):
    """Extended eval with per-class distribution + per-taxon AUC."""
    base = full_eval(y_true, y_pred, ns22_mask, taxon_masks)
    cls_dist = per_class_distribution(y_true, y_pred, mask=ns22_mask)
    base["per_class_dist"] = {k: v for k, v in cls_dist.items() if k != "auc_array"}
    base["per_class_auc"] = cls_dist.get("auc_array", [])
    return base

print("OK eval helpers (with rich_eval + per_class_distribution)")


OK eval helpers (with rich_eval + per_class_distribution)


In [8]:
# ============================================================
# Cell 8: train_r2 function
# ============================================================
def train_r2(fold_k, pseudo_meta_df, Y_pseudo, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"\n{'='*60}\n[Fold {fold_k} R2] train ({N_TOTAL_EPOCHS} ep)\n{'='*60}")

    items = []
    fds = FocalDS(train_df[train_df["fold"] != fold_k],
                  LABEL2IDX, secondary_lookup=focal_secondary_labels, aug=True)
    items.append(("focal", fds, len(fds)))
    if len(sc_meta) > 0:
        vm = sc_meta["fold"].values == fold_k
        sc_train_df = sc_meta[~vm].reset_index(drop=True)
        Y_tr = Y_SC[~vm]
        sds = LabeledSCDS(Y_tr, sc_train_df, aug=True)
        items.append(("labeled_sc", sds, len(sds)))
    pds = PseudoScDS(pseudo_meta_df, Y_pseudo, TS_DIR, aug=True)
    items.append(("pseudo_sc", pds, len(pds)))

    NAMES, DATASETS, SIZES = zip(*items)
    NAMES, DATASETS, SIZES = list(NAMES), list(DATASETS), list(SIZES)
    print(f"  Streams: {dict(zip(NAMES, SIZES))}")

    mds = ConcatDataset(DATASETS)
    n_steps_ep = max(100, int(sum(SIZES) / BATCH))
    print(f"  steps/epoch: {n_steps_ep}")

    model = make_model()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    scaler = GradScaler()
    warmup_steps = n_steps_ep * WARMUP_EPOCHS
    total_steps  = n_steps_ep * N_TOTAL_EPOCHS
    warmup_sched = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=1/25, end_factor=1.0, total_iters=warmup_steps)
    cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=total_steps - warmup_steps, eta_min=MIN_LR)
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer, schedulers=[warmup_sched, cosine_sched], milestones=[warmup_steps])

    if len(sc_meta) > 0:
        vm = sc_meta["fold"].values == fold_k
        val_sc_df = sc_meta[vm].reset_index(drop=True)
        Y_val = Y_SC[vm]
        ns22_val = non_s22_mask_sc[vm]
        val_wavs = _load_val_waveforms(val_sc_df)
    else:
        val_wavs = []
        Y_val = np.zeros((0, NUM_CLASSES), dtype=np.float32)
        ns22_val = np.zeros(0, dtype=bool)

    mel_transform = MelSpecTransform().to(device)
    spec_augment = SpecAugment().to(device)

    best_ns22 = -1.0
    best_macro = -1.0
    history = []
    epoch_times = []

    for epoch in range(N_TOTAL_EPOCHS):
        elapsed = time.time() - SESSION_START
        if epoch_times:
            est_next = max(epoch_times[-3:])
            if elapsed + est_next * 1.3 > MAX_RUNTIME_SEC:
                print(f"  [stop] time budget"); break
        ep_start = time.time()
        model.train()

        smp = MixSamp(SIZES, NAMES, SHARES_R2, BATCH, n_steps_ep, seed=42 + epoch)
        train_loader = DataLoader(
            mds, batch_sampler=smp, collate_fn=collate_m,
            num_workers=NUM_WORKERS,
            persistent_workers=PERSISTENT_WORKERS if NUM_WORKERS > 0 else False,
            pin_memory=True,
            prefetch_factor=4 if NUM_WORKERS > 0 else None,
        )

        el, el_cls, el_dist, nb_count = 0.0, 0.0, 0.0, 0
        for batch_idx, (wav, lb, perch_emb, wt, mk, sr) in enumerate(train_loader):
            wav = wav.to(device, non_blocking=True)
            lb = lb.to(device, non_blocking=True)
            perch_emb = perch_emb.to(device, non_blocking=True)
            wt = wt.to(device, non_blocking=True)
            mk = mk.to(device, non_blocking=True)
            sw = mk_sw(sr).to(device, non_blocking=True)

            with torch.no_grad():
                mel = mel_transform(wav)
                B = mel.size(0)
                for i in range(B):
                    mel[i] = (mel[i] - mel[i].mean()) / (mel[i].std() + 1e-6)
                mel = spec_augment(mel)
                mel = mel.to(memory_format=torch.channels_last)

            with autocast():
                clip_logits, framewise, distill_emb = model(
                    mel, return_framewise=True, return_distill=True)
                frame_max_logits = framewise.max(dim=1).values
                bce_clip = F.binary_cross_entropy_with_logits(clip_logits, lb, reduction="none")
                bce_frame = F.binary_cross_entropy_with_logits(frame_max_logits, lb, reduction="none")
                bce = 0.5 * bce_clip + 0.5 * bce_frame
                ps = (bce * wt * mk).sum(1) / (mk.sum(1) + 1e-8)
                cls_loss = (ps * sw).mean()
                distill_loss = F.mse_loss(distill_emb, perch_emb)
                loss = cls_loss + ALPHA_DISTILL * distill_loss

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            el += float(loss.item())
            el_cls += float(cls_loss.item())
            el_dist += float(distill_loss.item())
            nb_count += 1

            if batch_idx % 50 == 0:
                cur_lr = optimizer.param_groups[0]["lr"]
                print(f"    ep{epoch+1:02d} batch {batch_idx:4d}/{n_steps_ep}  "
                      f"loss={loss.item():.4f} cls={cls_loss.item():.4f} "
                      f"dist={distill_loss.item():.4f} lr={cur_lr:.2e}", flush=True)

        train_loss_avg = el / max(nb_count, 1)
        cls_loss_avg = el_cls / max(nb_count, 1)
        dist_loss_avg = el_dist / max(nb_count, 1)

        if len(val_wavs) > 0:
            val_preds = _predict_from_waveforms(model, mel_transform, val_wavs)
            r = rich_eval(Y_val, val_preds, ns22_val, TAXON_MASKS)
            val_ns22 = r["non_s22_macro"]
            val_macro = r["macro_auc_all"]
        else:
            val_ns22 = val_macro = float("nan")

        ep_elapsed = time.time() - ep_start
        epoch_times.append(ep_elapsed)
        ep = epoch + 1

        state_to_save = {
            "epoch": ep, "fold": fold_k, "round": "r2",
            "n_total_epochs": N_TOTAL_EPOCHS,
            "model_state": {k: v.cpu() for k, v in model.state_dict().items()},
            "best_ns22": best_ns22, "best_macro": best_macro,
        }
        torch.save(state_to_save, out_dir / "ckpt_latest.pth")
        if (not math.isnan(val_ns22)) and val_ns22 > best_ns22:
            best_ns22 = val_ns22
            state_to_save["best_ns22"] = best_ns22
            torch.save(state_to_save, out_dir / "ckpt_best_ns22.pth")
        if (not math.isnan(val_macro)) and val_macro > best_macro:
            best_macro = val_macro
            state_to_save["best_macro"] = best_macro
            torch.save(state_to_save, out_dir / "ckpt_best_macro.pth")

        history.append({"epoch": ep, "train_loss": round(train_loss_avg, 5),
                        "cls_loss": round(cls_loss_avg, 5),
                        "dist_loss": round(dist_loss_avg, 5),
                        "val_ns22": val_ns22, "val_macro": val_macro,
                        "per_taxon": r.get("per_taxon", {}),
                        "per_class_dist": r.get("per_class_dist", {}),
                        "lr": optimizer.param_groups[0]["lr"],
                        "elapsed_sec": round(ep_elapsed, 1)})
        with open(out_dir / "history.json", "w") as f:
            json.dump(history, f, indent=2, default=str)

        total_elapsed = time.time() - SESSION_START
        print(f"\n  === Ep {ep}/{N_TOTAL_EPOCHS}: loss={train_loss_avg:.4f} "
              f"cls={cls_loss_avg:.4f} dist={dist_loss_avg:.4f} "
              f"val_ns22={val_ns22:.4f} val_macro={val_macro:.4f} "
              f"({ep_elapsed:.0f}s, session {total_elapsed/60:.1f}min) ===\n")

        # ★ exp045: per-taxon + per-class distribution log (filter ablation diagnostic)
        if "per_taxon" in r:
            taxon_str = " ".join(f"{t}={a:.3f}" for t, a in r["per_taxon"].items())
            print(f"      taxon: {taxon_str}")
        if "per_class_dist" in r:
            d = r["per_class_dist"]
            if d.get("n_valid", 0) > 0:
                print(f"      class: n={d['n_valid']} median={d['median']:.3f} "
                      f"p25={d['p25']:.3f} p75={d['p75']:.3f} "
                      f"#>0.5={d['n_above_0.5']} #>0.7={d['n_above_0.7']} "
                      f"#>0.9={d['n_above_0.9']} #perfect={d['n_perfect_1.0']}")

        gc.collect()
        torch.cuda.empty_cache()

    print(f"  [R2 done] best_ns22={best_ns22:.4f}, best_macro={best_macro:.4f}")
    del optimizer, scheduler, scaler, mel_transform, spec_augment
    gc.collect(); torch.cuda.empty_cache()
    return best_ns22, best_macro

print("OK train_r2 ready")


OK train_r2 ready


In [9]:
# ============================================================
# Cell 9: Main 5-fold loop
# ============================================================
def mirror_dir_to_drive(local_dir, drive_dir):
    drive_dir.mkdir(parents=True, exist_ok=True)
    n = 0
    for f in sorted(local_dir.glob("*")):
        if not f.is_file(): continue
        if f.suffix not in {".pth", ".json"}: continue
        try:
            shutil.copy2(str(f), str(drive_dir / f.name))
            n += 1
        except Exception as e:
            print(f"    [WARN] {f.name}: {e}")
    return n


fold_results = {}

for FOLD_K in FOLDS:
    print(f"\n{'#'*70}\n# exp029 R3 (l1 student, e17 teacher) Fold {FOLD_K} (of {N_FOLDS}-fold split, single fold training)\n{'#'*70}")
    DRIVE_FOLD_DIR  = DRIVE_EXP_DIR / f"fold{FOLD_K}"
    # ★ exp029: teacher pseudo は exp017 R2 (pseudo_e17.csv) を Drive 上に置く前提
    # pseudo_e17.csv は exp028 で生成済 → Kaggle Notebook output から download → Drive へ
    # Drive path: kaggle/birdclef2026/pseudo_e17.csv (固定)
    DRIVE_TEACHER_PSEUDO = DRIVE_INPUT_DIR / "pseudo_e17.csv"
    DRIVE_R2_DIR    = DRIVE_FOLD_DIR / "r3"
    DRIVE_R2_DIR.mkdir(parents=True, exist_ok=True)

    if not DRIVE_TEACHER_PSEUDO.exists():
        print(f"  [SKIP] teacher pseudo missing: {DRIVE_TEACHER_PSEUDO}")
        print(f"  Download from Kaggle: maekeso/birdclef2026-exp028-pseudo-e17 (NB output)")
        print(f"  Or use kaggle api: api.kernels_output('maekeso/birdclef2026-exp028-pseudo-e17', path='/tmp')")
        fold_results[FOLD_K] = {"status": "skipped (no teacher pseudo)"}
        continue

    R2_BEST = DRIVE_R2_DIR / "ckpt_best_ns22.pth"
    if R2_BEST.exists():
        print(f"  Fold {FOLD_K}: R3 ckpt already on Drive — skip")
        try:
            st = torch.load(str(R2_BEST), map_location="cpu", weights_only=False)
            fold_results[FOLD_K] = {"r2_best_ns22": st.get("best_ns22", -1),
                                      "r2_best_macro": st.get("best_macro", -1),
                                      "status": "skipped"}
        except Exception:
            fold_results[FOLD_K] = {"status": "skipped (load err)"}
        continue

    LOCAL_R2 = LOCAL_OUT / f"fold{FOLD_K}" / "r3"
    LOCAL_R2.mkdir(parents=True, exist_ok=True)
    fold_t0 = time.time()

    print(f"  Loading teacher pseudo (exp017 R2): {DRIVE_TEACHER_PSEUDO}")
    pseudo_df_r1 = pd.read_csv(DRIVE_TEACHER_PSEUDO)
    print(f"    {len(pseudo_df_r1)} rows (raw)")

    # ★ exp045: paper 256 流 chunk filter (max_prob > threshold で chunk drop)
    if CHUNK_FILTER_MAX_PROB > 0:
        prob_cols = [c for c in pseudo_df_r1.columns if c in PRIMARY_LABELS]
        assert len(prob_cols) == NUM_CLASSES, f"Expected {NUM_CLASSES} prob cols, got {len(prob_cols)}"
        max_prob_per_row = pseudo_df_r1[prob_cols].max(axis=1)
        keep_mask = max_prob_per_row > CHUNK_FILTER_MAX_PROB
        n_before = len(pseudo_df_r1)
        n_after = int(keep_mask.sum())
        pseudo_df_r1 = pseudo_df_r1[keep_mask].reset_index(drop=True)
        print(f"    [chunk filter > {CHUNK_FILTER_MAX_PROB}] {n_before} -> {n_after} ({100*n_after/n_before:.1f}%)")
        print(f"      max_prob 50%ile={np.percentile(max_prob_per_row, 50):.3f} 90%ile={np.percentile(max_prob_per_row, 90):.3f}")
    else:
        print(f"    [chunk filter DISABLED] CHUNK_FILTER_MAX_PROB={CHUNK_FILTER_MAX_PROB}")

    Y_PSEUDO = pseudo_df_r1[PRIMARY_LABELS].values.astype(np.float32)
    pseudo_meta = pseudo_df_r1[["filename", "start_sec"]].copy().reset_index(drop=True)
    pseudo_meta["filename"] = pseudo_meta["filename"].astype(str)
    pseudo_meta["start_sec"] = pseudo_meta["start_sec"].astype(float)

    r2_best_ns22, r2_best_macro = train_r2(
        fold_k=FOLD_K, pseudo_meta_df=pseudo_meta, Y_pseudo=Y_PSEUDO,
        out_dir=LOCAL_R2)
    mirror_dir_to_drive(LOCAL_R2, DRIVE_R2_DIR)

    fold_elapsed = time.time() - fold_t0
    fold_results[FOLD_K] = {
        "r2_best_ns22": r2_best_ns22,
        "r2_best_macro": r2_best_macro,
        "elapsed_min": round(fold_elapsed / 60, 1),
    }
    print(f"\n  >>> Fold {FOLD_K} R2 DONE in {fold_elapsed/60:.1f}min: R2={r2_best_ns22:.4f} <<<\n")

    try:
        shutil.rmtree(LOCAL_R2, ignore_errors=True)
    except Exception: pass
    gc.collect(); torch.cuda.empty_cache()

print(f"\n{'='*60}\n5-fold R2 complete\n{'='*60}")
for k, v in sorted(fold_results.items()):
    print(f"  Fold {k}: {v}")



######################################################################
# exp029 R3 (l1 student, e17 teacher) Fold 0 (of 5-fold split, single fold training)
######################################################################
  Loading teacher pseudo (exp017 R2): /content/drive/MyDrive/kaggle/birdclef2026/pseudo_e17.csv
    127896 rows (raw)
    [chunk filter > 0.2] 127896 -> 84339 (65.9%)
      max_prob 50%ile=0.472 90%ile=0.965

[Fold 0 R2] train (20 ep)
  Streams: {'focal': 28906, 'labeled_sc': 584, 'pseudo_sc': 84339}
  steps/epoch: 592


model.safetensors:   0%|          | 0.00/166M [00:00<?, ?B/s]

    ep01 batch    0/592  loss=1.1006 cls=1.0693 dist=0.0313 lr=1.21e-05
    ep01 batch   50/592  loss=0.6543 cls=0.6433 dist=0.0111 lr=1.82e-05
    ep01 batch  100/592  loss=0.6180 cls=0.6070 dist=0.0110 lr=2.43e-05
    ep01 batch  150/592  loss=0.5605 cls=0.5499 dist=0.0106 lr=3.04e-05
    ep01 batch  200/592  loss=0.4450 cls=0.4349 dist=0.0101 lr=3.64e-05
    ep01 batch  250/592  loss=0.3571 cls=0.3473 dist=0.0098 lr=4.25e-05
    ep01 batch  300/592  loss=0.3011 cls=0.2915 dist=0.0095 lr=4.86e-05
    ep01 batch  350/592  loss=0.2714 cls=0.2618 dist=0.0096 lr=5.47e-05
    ep01 batch  400/592  loss=0.2073 cls=0.1983 dist=0.0091 lr=6.08e-05
    ep01 batch  450/592  loss=0.1784 cls=0.1701 dist=0.0084 lr=6.69e-05
    ep01 batch  500/592  loss=0.1345 cls=0.1259 dist=0.0087 lr=7.29e-05
    ep01 batch  550/592  loss=0.0884 cls=0.0797 dist=0.0086 lr=7.90e-05

  === Ep 1/20: loss=0.3638 cls=0.3539 dist=0.0099 val_ns22=0.6696 val_macro=0.6696 (218s, session 4.1min) ===

      taxon: Aves=0.483 

In [10]:
# ============================================================
# Cell 10: Upload R3 ckpts to Kaggle Dataset (robust try-version-first pattern)
# ============================================================
import tempfile

KJ = next((p for p in [DRIVE_INPUT_DIR / "kaggle.json",
                        Path("/content/drive/MyDrive/kaggle.json")] if p.exists()), None)
if KJ is not None:
    KAGGLE_CFG = Path.home() / ".kaggle"
    KAGGLE_CFG.mkdir(parents=True, exist_ok=True)
    shutil.copy(str(KJ), str(KAGGLE_CFG / "kaggle.json"))
    os.chmod(str(KAGGLE_CFG / "kaggle.json"), 0o600)
    creds = json.loads(KJ.read_text())
    if creds.get("key", "").startswith("KGAT_"):
        os.environ["KAGGLE_API_TOKEN"] = creds["key"]
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
print("kaggle re-auth OK")

USER  = "maekeso"
SLUG  = "birdclef2026-exp045-l1-filtered"
TITLE = "birdclef2026 exp045 l1 filtered ablation"

with tempfile.TemporaryDirectory() as td:
    td = Path(td)
    n_staged = 0
    for fold_k in FOLDS:
        # ★ exp045: filter ablation R3 ckpt のみ upload
        DRIVE_R3_DIR = DRIVE_EXP_DIR / f"fold{fold_k}" / "r3"
        for fn in ["ckpt_best_ns22.pth", "ckpt_best_macro.pth", "history.json"]:
            src = DRIVE_R3_DIR / fn
            if src.exists():
                dst = td / f"r3_fold{fold_k}_{fn}"
                shutil.copy2(str(src), str(dst))
                n_staged += 1
                print(f"  staged: {dst.name} ({src.stat().st_size/1e6:.1f} MB)")
    print(f"Staged {n_staged} files total")
    assert n_staged > 0, "No files to upload — check DRIVE_EXP_DIR / fold paths"

    meta = {"title": TITLE, "id": f"{USER}/{SLUG}",
            "licenses": [{"name": "CC0-1.0"}]}
    (td / "dataset-metadata.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
    summary = " ; ".join([f"f{k}_r3={v.get('r2_best_ns22','?')}" for k, v in sorted(fold_results.items())])
    version_notes = f"exp045 l1 chunk-filter ablation (filter={CHUNK_FILTER_MAX_PROB}) | {summary}"[:498]

    # ★ robust: try-version-first, fallback to create_new on failure (no pre-check!)
    # pre-check (dataset_view/list_files) is unreliable for auth quirks/rate limits
    upload_method = None
    upload_error = None
    try:
        print(f"\nAttempting dataset_create_version for {USER}/{SLUG} ...")
        api.dataset_create_version(folder=str(td), version_notes=version_notes,
                                    dir_mode="zip", quiet=False)
        upload_method = "version_up"
        print(f"OK uploaded as new version")
    except Exception as e:
        upload_error = str(e)[:300]
        print(f"  version_create failed: {upload_error}")
        print(f"  → fallback to dataset_create_new...")
        try:
            api.dataset_create_new(folder=str(td), public=False,
                                    dir_mode="zip", quiet=False)
            upload_method = "create_new"
            print(f"OK created new dataset")
        except Exception as e2:
            upload_method = "FAILED"
            print(f"  create_new ALSO failed: {str(e2)[:300]}")

    # ★ strong verification: file count check (catches silent fail)
    print(f"\nVerifying upload (method={upload_method})...")
    try:
        import time
        time.sleep(5)  # give Kaggle time to register
        files_after = api.dataset_list_files(f"{USER}/{SLUG}").files
        n_uploaded = len(files_after)
        print(f"  remote files: {n_uploaded}")
        for f in files_after[:10]:
            print(f"    - {f.name}")
        if n_uploaded < n_staged:
            print(f"  ⚠ WARN: only {n_uploaded} files visible, staged {n_staged}")
            print(f"  ⚠ Possible silent fail. Re-run upload cell or check Kaggle UI")
        else:
            print(f"  ✓ verified: {n_uploaded} >= {n_staged} (staged)")
    except Exception as e:
        print(f"  [VERIFY FAIL] {str(e)[:300]}")
        print(f"  → Cannot list files. Check Kaggle UI: https://www.kaggle.com/datasets/{USER}/{SLUG}")

    print(f"\nURL: https://www.kaggle.com/datasets/{USER}/{SLUG}")
    print(f"Upload method: {upload_method}")
    if upload_error:
        print(f"Initial error (recovered if method != FAILED): {upload_error}")


kaggle re-auth OK
  staged: r3_fold0_ckpt_best_ns22.pth (179.6 MB)
  staged: r3_fold0_ckpt_best_macro.pth (179.6 MB)
  staged: r3_fold0_history.json (0.0 MB)
Staged 3 files total

Attempting dataset_create_version for maekeso/birdclef2026-exp045-l1-filtered ...
Starting upload for file r3_fold0_ckpt_best_macro.pth


100%|██████████| 171M/171M [00:05<00:00, 34.9MB/s]


Upload successful: r3_fold0_ckpt_best_macro.pth (171MB)
Starting upload for file r3_fold0_history.json


100%|██████████| 12.8k/12.8k [00:00<00:00, 32.7kB/s]


Upload successful: r3_fold0_history.json (13KB)
Starting upload for file r3_fold0_ckpt_best_ns22.pth


...

Upload successful: r3_fold0_ckpt_best_macro.pth (171MB)
Starting upload for file r3_fold0_history.json
Error while trying to load upload info: KaggleObject.from_dict() got an unexpected keyword argument 'token'


100%|██████████| 12.8k/12.8k [00:00<00:00, 31.9kB/s]


Upload successful: r3_fold0_history.json (13KB)
Starting upload for file r3_fold0_ckpt_best_ns22.pth
Error while trying to load upload info: KaggleObject.from_dict() got an unexpected keyword argument 'token'


100%|██████████| 171M/171M [00:05<00:00, 34.9MB/s]


Upload successful: r3_fold0_ckpt_best_ns22.pth (171MB)
OK created new dataset

Verifying upload (method=create_new)...
  remote files: 3
    - r3_fold0_ckpt_best_macro.pth
    - r3_fold0_ckpt_best_ns22.pth
    - r3_fold0_history.json
  ✓ verified: 3 >= 3 (staged)

URL: https://www.kaggle.com/datasets/maekeso/birdclef2026-exp045-l1-filtered
Upload method: create_new
Initial error (recovered if method != FAILED): 403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/CreateDatasetVersion


In [11]:
# ============================================================
# Cell 11: Summary
# ============================================================
total_time = time.time() - SESSION_START
print(f"\n{'='*60}")
print(f"exp029 l1 single fold R3 summary")
print(f"{'='*60}")
print(f"  Backbone: {BACKBONE}")
print(f"  R2 epochs/fold: {N_TOTAL_EPOCHS}")
print(f"  Total time: {total_time/60:.1f} min ({total_time/3600:.2f}h)")
for k, v in sorted(fold_results.items()):
    print(f"  Fold {k}: {v}")
print(f"\n  Kaggle Dataset: https://www.kaggle.com/datasets/maekeso/birdclef2026-exp029-l1-single")



exp029 l1 single fold R3 summary
  Backbone: eca_nfnet_l1
  R2 epochs/fold: 20
  Total time: 68.4 min (1.14h)
  Fold 0: {'r2_best_ns22': 0.9347, 'r2_best_macro': 0.9347, 'elapsed_min': 67.6}

  Kaggle Dataset: https://www.kaggle.com/datasets/maekeso/birdclef2026-exp029-l1-single


In [12]:
# ============================================================
# Cell 12: Terminate Colab runtime
# ============================================================
print("All R2 phases complete. Terminating Colab runtime in 5s...")
import time as _t
_t.sleep(5)
from google.colab import runtime
runtime.unassign()


All R2 phases complete. Terminating Colab runtime in 5s...
